### 🛠️ Initialize Notebook Variables

**Only modify entries under _USER CONFIGURATION_.**

In [ ]:
import utils
from pathlib import Path
from typing import List

from apimtypes import API, APIM_SKU, GET_APIOperation2, INFRASTRUCTURE, Region
from console import print_ok, print_val
from azure_resources import get_account_info, get_infra_rg_name

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = Region.EAST_US_2
index = 1
apim_sku = APIM_SKU.BASICV2  # Options: 'DEVELOPER', 'BASIC', 'STANDARD', 'PREMIUM', 'BASICV2', 'STANDARDV2', 'PREMIUMV2'
deployment = INFRASTRUCTURE.APIM_ACA  # Options: see supported_infras below
api_prefix = 'sse-'  # ENTER A PREFIX FOR THE APIS TO REDUCE COLLISION POTENTIAL WITH OTHER SAMPLES
tags = ['sse', 'streaming']  # ENTER DESCRIPTIVE TAG(S)

# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder = 'sse'
rg_name = get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.AFD_APIM_PE,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
]

nb_helper = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index=index,
    apim_sku=apim_sku,
)

# Get account info (subscription/tenant) for display + later use
current_user, current_user_id, tenant_id, subscription_id = get_account_info()

backend_id = 'sse-backend'

pol_sse_good = utils.read_policy_xml('sse-good.xml', sample_name=sample_folder).format(backend_id=backend_id)
pol_sse_buffered = utils.read_policy_xml('sse-buffered.xml', sample_name=sample_folder).format(backend_id=backend_id)
pol_sse_health = utils.read_policy_xml('sse-health.xml', sample_name=sample_folder).format(backend_id=backend_id)

api_path = f'{api_prefix}sse'

op_good = GET_APIOperation2(
    'good',
    'Good (unbuffered)',
    '/good',
    'SSE pass-through with buffer-response="false"',
    pol_sse_good,
)

op_buffered = GET_APIOperation2(
    'buffered',
    'Buffered (bad example)',
    '/buffered',
    'SSE with buffer-response="true" (intentionally delays chunks)',
    pol_sse_buffered,
)

op_health = GET_APIOperation2(
    'health',
    'Health',
    '/health',
    'Health probe to the backend (non-streaming)',
    pol_sse_health,
)

sse_api = API(
    api_path,
    'SSE Demo API',
    api_path,
    'Demonstrates SSE behavior through APIM with buffering on vs off',
    operations=[op_health, op_good, op_buffered],
    tags=tags,
    subscriptionRequired=True,
)

apis: List[API] = [sse_api]

print_val('Resource group', rg_name)
print_ok('Notebook initialized')

### 🐳 Deploy the SSE Backend (Azure Container Apps)

This deploys the sample backend from `samples/sse/backend/` as a public Container App and captures its FQDN for use as the APIM backend URL.

❗️This notebook assumes the infrastructure resource group already exists. If it doesn't, run an infrastructure `create.ipynb` first.

In [ ]:
from pathlib import Path

from azure_resources import does_resource_group_exist, run
from console import print_error, print_message, print_ok, print_val, print_warning

# Pre-check: infrastructure resource group must exist
if not does_resource_group_exist(rg_name):
    print_error(f'Resource group does not exist: {rg_name}')
    raise SystemExit('Create the infrastructure first, then re-run this cell.')

# Find an existing Container Apps Environment in this infra RG
env_list = run(
    f'az containerapp env list --resource-group {rg_name} -o json',
    error_message='Failed to query Container Apps environments',
)
if not env_list.success or not isinstance(env_list.json_data, list) or not env_list.json_data:
    raise SystemExit(
        'No Container Apps environment found in this infrastructure. '
        'Choose a supported infrastructure (e.g. APIM_ACA).',
    )

envs = env_list.json_data
env_name = None
for env in envs:
    if isinstance(env, dict) and isinstance(env.get('name'), str) and env['name'].startswith('cae-'):
        env_name = env['name']
        break
if env_name is None:
    env_name = envs[0].get('name')

print_val('Container Apps environment', env_name)

# Deterministic name: re-runs will update in-place
ca_name = f'ca-sse-backend-{index}'
project_root = Path(utils.find_project_root())
backend_folder = (project_root / 'samples' / sample_folder / 'backend').resolve()

print_message('Deploying backend (this can take a few minutes)...', blank_above=True)

# Note: some Azure CLI versions do not allow `-o json` for `az containerapp up`.
up = run(
    f'az containerapp up --resource-group {rg_name} --name {ca_name} --environment {env_name} '
    f'--source "{backend_folder}" --ingress external --target-port 8000',
    ok_message='Backend deployed',
    error_message='Backend deployment failed',
)

# `az containerapp up` may return non-zero even though the app ends up created/updated.
# If it fails, fall back to querying the app and proceed if it's actually there.
if not up.success:
    print_warning('`az containerapp up` reported failure; checking whether the app exists anyway...')

# Fetch backend FQDN after deployment
fqdn_out = run(
    f'az containerapp show --resource-group {rg_name} --name {ca_name} '
    ' --query properties.configuration.ingress.fqdn -o tsv',
    error_message='Failed to query backend FQDN',
)

fqdn = fqdn_out.text.strip() if fqdn_out.success else None

if not fqdn:
    print_warning('Could not determine backend FQDN automatically')
    raise SystemExit(1)

# Validate the app is actually running the sample backend image (not the default quickstart)
image_out = run(
    f'az containerapp show --resource-group {rg_name} --name {ca_name} '
    ' --query properties.template.containers[0].image -o tsv',
    error_message='Failed to query backend container image',
)
image = image_out.text.strip() if image_out.success else ''
if image:
    print_val('Backend image', image)

if 'mcr.microsoft.com/k8se/quickstart' in image:
    print_error(
        'Backend container app is still running the default quickstart image. '
        'Re-run this cell to build and deploy the SSE backend image.',
    )
    raise SystemExit(1)

backend_url = f'https://{fqdn}'
print_val('Backend URL', backend_url)
print_ok('Backend ready')

### 🚀 Deploy APIs

Deploys the APIM backend + API/operations/policies in `samples/sse/main.bicep` into the previously-specified resource group.

In [ ]:
from azure_resources import run
from console import print_error, print_ok, print_val, print_warning

# Allow skipping the backend deployment cell if the backend already exists.
# If backend_url is missing, try to discover it from the deterministic Container App name.
if 'backend_url' not in locals() or not isinstance(backend_url, str) or not backend_url.strip():
    ca_name = f'ca-sse-backend-{index}'
    print_warning(
        'backend_url is not set in this kernel session. '
        f'Trying to discover it from existing Container App: {ca_name}',
    )
    fqdn_out = run(
        f'az containerapp show --resource-group {rg_name} --name {ca_name} '
        ' --query properties.configuration.ingress.fqdn -o tsv',
        error_message='Failed to query existing backend FQDN',
    )
    fqdn = fqdn_out.text.strip() if fqdn_out.success else ''
    if fqdn:
        backend_url = f'https://{fqdn}'
        print_val('Discovered backend_url', backend_url)
    else:
        raise SystemExit(
            'backend_url is not set and could not be discovered. '
            'Run Cell 4 (Deploy the SSE Backend), or ensure the existing backend Container App exists '
            f'with name {ca_name} in resource group {rg_name}.',
        )

# Build the bicep parameters
bicep_parameters = {
    'apis': {'value': [api.to_dict() for api in apis]},
    'backendUrl': {'value': backend_url},
}

# Deploy the sample
output = nb_helper.deploy_sample(bicep_parameters)

if output.success:
    apim_name = output.get('apimServiceName', 'APIM Service Name')
    apim_gateway_url = output.get('apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_apis = output.getJson('apiOutputs', 'APIs')

    # Keep locals consistent if NotebookHelper selected a different infra
    deployment = nb_helper.deployment
    rg_name = nb_helper.rg_name

    print_ok('Deployment completed successfully')
else:
    print_error('Deployment failed!')
    raise SystemExit(1)

### ✅ Test SSE Behavior (Good vs Buffered)

This calls the APIM operations and measures time-to-first-event.

- **Good** should deliver the first event quickly.
- **Buffered** should delay delivery until APIM's buffer fills or the stream ends.
- You can run this section after a kernel restart: the test cell will try to re-initialize defaults and discover the APIM gateway URL + a usable subscription key via Azure CLI.

In [ ]:
%pip -q install requests

In [14]:
import time
from typing import Optional

import utils
from apimrequests import ApimRequests
from apimtesting import ApimTesting
from apimtypes import APIM_SKU, INFRASTRUCTURE, Region
from azure_resources import get_apim_subscription_key, get_apim_url, get_infra_rg_name, run
from console import print_val

# -----------------------------------------------------------------------------
# Minimal self-healing setup
# -----------------------------------------------------------------------------
# If you run this cell in a fresh kernel, try to recreate the key notebook variables
# and discover the APIM gateway URL + a valid subscription key using Azure CLI.

try:
    rg_location
except NameError:
    rg_location = Region.EAST_US_2

try:
    index
except NameError:
    index = 1

try:
    apim_sku
except NameError:
    apim_sku = APIM_SKU.BASICV2

try:
    deployment
except NameError:
    deployment = INFRASTRUCTURE.APIM_ACA

try:
    api_prefix
except NameError:
    api_prefix = 'sse-'

try:
    sample_folder
except NameError:
    sample_folder = 'sse'

try:
    rg_name
except NameError:
    rg_name = get_infra_rg_name(deployment, index)

try:
    api_path
except NameError:
    api_path = f'{api_prefix}sse'

# NotebookHelper is used by shared endpoint resolution logic (AFD/AppGW/APIM).
try:
    supported_infras
except NameError:
    supported_infras = [
        INFRASTRUCTURE.AFD_APIM_PE,
        INFRASTRUCTURE.APIM_ACA,
        INFRASTRUCTURE.APPGW_APIM,
        INFRASTRUCTURE.APPGW_APIM_PE,
    ]

try:
    nb_helper
except NameError:
    nb_helper = utils.NotebookHelper(
        sample_folder,
        rg_name,
        rg_location,
        deployment,
        supported_infras,
        index=index,
        apim_sku=apim_sku,
    )

# -----------------------------------------------------------------------------
# Discover APIM gateway URL + subscription key (if deployment outputs are missing)
# -----------------------------------------------------------------------------
try:
    apim_gateway_url
except NameError:
    apim_gateway_url = ''

if not isinstance(apim_gateway_url, str) or not apim_gateway_url.strip():
    apim_gateway_url = get_apim_url(rg_name)
    if not apim_gateway_url:
        raise SystemExit(
            f'Could not discover APIM gateway URL in resource group: {rg_name}. '
            'Confirm the infrastructure exists and you are logged in to Azure CLI.',
        )

# Try to obtain a subscription key. Prefer the deployment output when present, otherwise discover via ARM.
subscription_key: Optional[str] = None

try:
    apim_apis
except NameError:
    apim_apis = None

if isinstance(apim_apis, list) and apim_apis:
    first = apim_apis[0]
    if isinstance(first, dict):
        subscription_key = first.get('subscriptionPrimaryKey')

if not subscription_key:
    # Resolve APIM name from the infra resource group (assumes first APIM in that RG is the target).
    apim_name_out = run(f'az apim list -g {rg_name} --query "[0].name" -o tsv', log_command=False)
    apim_name = apim_name_out.text.strip() if apim_name_out.success else ''
    if not apim_name:
        raise SystemExit(f'Could not resolve APIM name in resource group: {rg_name}')

    sub_id_out = run('az account show --query id -o tsv', log_command=False)
    subscription_id = sub_id_out.text.strip() if sub_id_out.success else ''
    if not subscription_id:
        raise SystemExit('Could not resolve Azure subscription id (are you logged in to Azure CLI?)')

    api_version = '2022-08-01'
    apim_arm_base = (
        f'https://management.azure.com/subscriptions/{subscription_id}'
        f'/resourceGroups/{rg_name}/providers/Microsoft.ApiManagement/service/{apim_name}'
    )

    # Find the API resource name by matching the APIM API path
    api_name: Optional[str] = None
    apis_out = run(
        f'az rest --method get --url "{apim_arm_base}/apis?api-version={api_version}" -o json',
        log_command=False,
    )
    if apis_out.success and isinstance(apis_out.json_data, dict):
        value = apis_out.json_data.get('value')
        if isinstance(value, list):
            for item in value:
                if not isinstance(item, dict):
                    continue
                props = item.get('properties') or {}
                if isinstance(props, dict) and props.get('path') == api_path:
                    api_name = str(item.get('name') or '').strip() or None
                    break

    # Find an active subscription scoped to this API (if such a subscription exists)
    sid: Optional[str] = None
    if api_name:
        subs_out = run(
            f'az rest --method get --url "{apim_arm_base}/subscriptions?api-version={api_version}" -o json',
            log_command=False,
        )
        if subs_out.success and isinstance(subs_out.json_data, dict):
            value = subs_out.json_data.get('value')
            if isinstance(value, list):
                for s in value:
                    if not isinstance(s, dict):
                        continue
                    props = s.get('properties') or {}
                    if not isinstance(props, dict):
                        continue
                    state = str(props.get('state') or '').lower()
                    scope = str(props.get('scope') or '')
                    if state == 'active' and f'/apis/{api_name}' in scope:
                        sid = str(s.get('name') or '').strip() or None
                        break

    subscription_key = get_apim_subscription_key(
        apim_name,
        rg_name,
        subscription_id=subscription_id,
        sid=sid,
    )

if not subscription_key or not isinstance(subscription_key, str) or not subscription_key.strip():
    raise SystemExit('Could not determine a usable subscription key for this APIM instance.')

# -----------------------------------------------------------------------------
# Execute tests + print elapsed time for each call
# -----------------------------------------------------------------------------
tests = ApimTesting('SSE Sample Tests', sample_folder, nb_helper.deployment)

endpoint_url, request_headers = utils.get_endpoint(nb_helper.deployment, nb_helper.rg_name, apim_gateway_url)
reqs = ApimRequests(endpoint_url, subscription_key, request_headers)

print_val('Endpoint URL', endpoint_url)

# Health
start = time.perf_counter()
health = reqs.singleGet(f'/{api_path}/health', msg='Calling health endpoint')
health_elapsed_s = time.perf_counter() - start
print_val('Health elapsed_s', f'{health_elapsed_s:.3f}')
tests.verify(health is not None and 'ok' in str(health), True, label='Health returned ok')

# Good streaming
start = time.perf_counter()
good = reqs.sseGet(
    f'/{api_path}/good?interval_ms=200&total_events=50&heartbeat_ms=1000',
    msg='Calling GOOD (unbuffered) SSE endpoint',
    max_events=3,
    max_seconds=10,
)
good_elapsed_s = time.perf_counter() - start
print_val('Good time_to_first_event_s', good.get('time_to_first_event_s'))
print_val('Good elapsed_s', f'{good_elapsed_s:.3f}')
tests.verify(
    good.get('time_to_first_event_s') is not None and good['time_to_first_event_s'] < 1.5,
    True,
    label='Good: first event quickly',
)

# Buffered streaming (intentionally bad example)
start = time.perf_counter()
buffered = reqs.sseGet(
    f'/{api_path}/buffered?interval_ms=50&total_events=250&heartbeat_ms=0',
    msg='Calling BUFFERED SSE endpoint',
    max_events=3,
    max_seconds=15,
)
buffered_elapsed_s = time.perf_counter() - start
print_val('Buffered time_to_first_event_s', buffered.get('time_to_first_event_s'))
print_val('Buffered elapsed_s', f'{buffered_elapsed_s:.3f}')
tests.verify(
    buffered.get('time_to_first_event_s') is None or buffered['time_to_first_event_s'] > 2.0,
    True,
    label='Buffered: first event delayed (or not seen within timeout)',
)

tests.print_summary()


ℹ️ Identifying possible endpoints for infrastructure apim-aca... ⌚ 10:23:25.529242
⚠️ No Front Door endpoint URL found. ⌚ 10:23:25.530023
✅ APIM Service Name: apim-ugzlw22ihnb6g ⌚ 10:23:27.600856
✅ APIM Gateway URL: https://apim-ugzlw22ihnb6g.azure-api.net ⌚ 10:23:27.601974

ℹ️ Checking if the infrastructure architecture deployment uses Azure Front Door. ⌚ 10:23:30.475237
⚠️ No Front Door endpoint URL found. ⌚ 10:23:30.477024

ℹ️ Using APIM Gateway URL: https://apim-ugzlw22ihnb6g.azure-api.net ⌚ 10:23:30.478000
👉 Endpoint URL             : https://apim-ugzlw22ihnb6g.azure-api.net

ℹ️ Calling health endpoint ⌚ 10:23:30.480938
ℹ️ GET https://apim-ugzlw22ihnb6g.azure-api.net/sse-sse/health
ℹ️ {'api-key': '1e99d669bc614b0d988e7daf996dd28a', 'Accept': 'application/json'}
👉 Response status          : 200 - OK
👉 Response headers         :
{'Content-Length': '39', 'Content-Type': 'application/json', 'Date': 'Fri, 06 Mar 2026 16:23:29 GMT', 'X-Backend-URL':
'https://ca-sse-backend-1.ashymeadow-

🔍 Assert that Health returned ok value [True] matches expected value [True].
✅ Test 1: PASS


👉 Response status          : 200 - OK
👉 Good time_to_first_event_s: 0.2504105567932129
👉 Good elapsed_s           : 0.652

ℹ️ Calling BUFFERED SSE endpoint ⌚ 10:23:31.436728
ℹ️ GET https://apim-ugzlw22ihnb6g.azure-api.net/sse-sse/buffered?interval_ms=50&total_events=250&heartbeat_ms=0 (SSE)
ℹ️ {'api-key': '1e99d669bc614b0d988e7daf996dd28a', 'Accept': 'text/event-stream'}


🔍 Assert that Good: first event quickly value [True] matches expected value [True].
✅ Test 2: PASS


👉 Response status          : 200 - OK
👉 Buffered time_to_first_event_s: 12.778940439224243
👉 Buffered elapsed_s       : 12.782


🔍 Assert that Buffered: first event delayed (or not seen within timeout) value [True] matches expected value [True].
✅ Test 3: PASS


             🧪 SSE Sample Tests - Test Results Summary 🧪

 Sample Name : sse
 Deployment  : APIM_ACA

📊 Test Execution Statistics:
    • Total Tests  :     3
    • Tests Passed :     3
    • Tests Failed :     0 
    • Success Rate : 100.0%

🎉 OVERALL RESULT: ALL TESTS PASSED! 🎉
✨ Congratulations! Your APIM deployment is working flawlessly! ✨


              🎯 Test execution completed successfully! 🎯              



In [ ]:
import json
import os
import shutil
import subprocess
from typing import Any

from apimtypes import SUBSCRIPTION_KEY_PARAMETER_NAME
from console import print_error, print_message, print_ok, print_val, print_warning

# Prereqs from previous cells
required_vars = ['endpoint_url', 'request_headers', 'subscription_key', 'api_path']
missing = [v for v in required_vars if v not in locals()]
if missing:
    raise SystemExit(f'Missing notebook variables (run previous cells first): {missing}')

if not isinstance(subscription_key, str) or not subscription_key.strip():
    raise SystemExit('subscription_key is missing/invalid; re-run the deployment cell.')

if not isinstance(endpoint_url, str) or not endpoint_url.strip():
    raise SystemExit('endpoint_url is missing/invalid; re-run the deployment/test initialization cells.')

if not shutil.which('curl'):
    raise SystemExit('curl was not found on PATH. Install curl and re-run this cell.')

def _run_curl_timing(*, url: str, headers: dict[str, str], timeout_s: int = 30) -> dict[str, Any]:
    # `time_starttransfer` ~= time to first byte (TTFB). For SSE, this is the most useful signal.
    # We discard the body to keep notebook output clean.
    devnull = os.devnull
    write_out = (
        '{'
        '"http_code":%{http_code},'
        '"time_starttransfer":%{time_starttransfer},'
        '"time_total":%{time_total},'
        '"size_download":%{size_download}'
        '}'
    )

    args: list[str] = [
        'curl',
        '-sS',
        '-N',  # no buffering on the client side
        '--http1.1',
        '-k',  # allow self-signed certs in some infra paths
        '--max-time',
        str(timeout_s),
        '-o',
        devnull,
        '-w',
        write_out,
    ]

    for name, value in headers.items():
        args.extend(['-H', f'{name}: {value}'])

    args.append(url)

    proc = subprocess.run(args, capture_output=True, text=True, check=False)
    out = (proc.stdout or '').strip()

    if proc.returncode != 0:
        err = (proc.stderr or '').strip()
        raise RuntimeError(f'curl failed (exit {proc.returncode}). stdout={out!r} stderr={err!r}')

    try:
        metrics = json.loads(out)
    except json.JSONDecodeError as e:
        raise RuntimeError(f'Unexpected curl output (expected JSON): {out!r}') from e

    return metrics

base = endpoint_url.rstrip('/')
good_url = f'{base}/{api_path}/good?interval_ms=200&total_events=20&heartbeat_ms=1000'
bad_url = f'{base}/{api_path}/buffered?interval_ms=50&total_events=80&heartbeat_ms=0'

common_headers: dict[str, str] = {}
if isinstance(request_headers, dict):
    for k, v in request_headers.items():
        if isinstance(k, str) and isinstance(v, str) and v:
            common_headers[k] = v

common_headers[SUBSCRIPTION_KEY_PARAMETER_NAME] = subscription_key
common_headers['Accept'] = 'text/event-stream'

print_message('curl timing test (GOOD vs BAD buffering)', blank_above=True)
print_val('GOOD URL', good_url)
print_val('BAD URL', bad_url)

try:
    good = _run_curl_timing(url=good_url, headers=common_headers, timeout_s=20)
    bad = _run_curl_timing(url=bad_url, headers=common_headers, timeout_s=20)
except Exception as e:
    print_error(str(e))
    raise

good_ttfb_s = float(good.get('time_starttransfer') or 0.0)
good_total_s = float(good.get('time_total') or 0.0)
bad_ttfb_s = float(bad.get('time_starttransfer') or 0.0)
bad_total_s = float(bad.get('time_total') or 0.0)

print_val('GOOD time_starttransfer_s (TTFB)', f'{good_ttfb_s:.3f}')
print_val('GOOD time_total_s', f'{good_total_s:.3f}')
print_val('BAD  time_starttransfer_s (TTFB)', f'{bad_ttfb_s:.3f}')
print_val('BAD  time_total_s', f'{bad_total_s:.3f}')

delta_ttfb_s = bad_ttfb_s - good_ttfb_s
print_val('Δ TTFB (BAD - GOOD) seconds', f'{delta_ttfb_s:.3f}')

if good_ttfb_s > 1.5:
    print_warning('GOOD call TTFB is higher than expected; check for buffering/logging upstream.')
if bad_ttfb_s < 2.0:
    print_warning('BAD call TTFB is lower than expected; buffering might not be taking effect in this path.')

print_ok('curl timing test completed')